In [6]:
!pip install ccxt==4.3.90 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.3/118.3 kB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 188.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 123.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 388.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.7/153.7 kB 326.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 266.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 142.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 379.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 174.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 kB 314.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 272.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.6/219.6 kB 316.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [14]:
!pip install ccxt pandas requests matplotlib seaborn scipy


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
import ccxt
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np
from matplotlib.patches import Patch

plt.rcParams['axes.unicode_minus'] = False

# =====================
# 1. Kraken Price Data
# =====================
exchange = ccxt.kraken()
ohlcv = exchange.fetch_ohlcv('BTC/USDT', timeframe='1d', limit=365)
df_price = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
df_price['date'] = pd.to_datetime(df_price['timestamp'], unit='ms').dt.date
df_price = df_price[['date', 'close']]
print("✅ Kraken price data loaded")

# =====================
# 2. Fear & Greed Data
# =====================
url = "https://api.alternative.me/fng/?limit=365&format=json"
fng_data = requests.get(url).json()['data']
df_fng = pd.DataFrame(fng_data)
df_fng['date'] = pd.to_datetime(df_fng['timestamp'], unit='s').dt.date
df_fng['fng_value'] = df_fng['value'].astype(int)
df_fng = df_fng[['date', 'fng_value']].sort_values('date').reset_index(drop=True)
print("✅ Fear & Greed data loaded")

# =====================
# 3. Merge & Returns
# =====================
df = pd.merge(df_price, df_fng, on='date', how='inner')
df['return_1d'] = df['close'].pct_change() * 100
df['return_3d'] = df['close'].pct_change(3) * 100
df['return_7d'] = df['close'].pct_change(7) * 100
df = df.dropna().reset_index(drop=True)
print(f"✅ Merged - Total {len(df)} days\n")

# =====================
# 4. Correlation Output
# =====================
print("=" * 58)
print("📊 FNG <-> BTC Return Correlation Analysis")
print("=" * 58)

for ret_col, label in [('return_1d', '1D Return'), ('return_3d', '3D Return'), ('return_7d', '7D Return')]:
    r, p = stats.pearsonr(df['fng_value'], df[ret_col])
    sig = "✅ Significant" if p < 0.05 else "❌ Not Significant"
    print(f"FNG <-> BTC {label}: r = {r:+.4f}, p = {p:.4f}  {sig}")

print()
print("=" * 58)
print("📊 Lead-Lag Analysis: FNG -> BTC 1D Return")
print("=" * 58)
print(f"{'Lag(days)':<12} {'r':>8} {'p-value':>10} {'Sig':>6}  {'Interpretation'}")
print("-" * 58)

lags = range(-7, 8)
lag_results = []
for lag in lags:
    if lag < 0:
        x = df['fng_value'].iloc[:lag].values
        y = df['return_1d'].iloc[-lag:].values
        direction = f"FNG leads by {abs(lag)}d"
    elif lag > 0:
        x = df['fng_value'].iloc[lag:].values
        y = df['return_1d'].iloc[:-lag].values
        direction = f"Price leads by {lag}d"
    else:
        x = df['fng_value'].values
        y = df['return_1d'].values
        direction = "Concurrent"

    r, p = stats.pearsonr(x, y)
    sig = "✅" if p < 0.05 else "  "
    lag_results.append({'lag': lag, 'r': r, 'p': p, 'direction': direction})
    print(f"{lag:<12} {r:>+8.4f} {p:>10.4f} {sig:>6}  {direction}")

df_lag = pd.DataFrame(lag_results)
best = df_lag.loc[df_lag['r'].abs().idxmax()]
print(f"\n📌 Max correlation: lag={int(best['lag'])}d ({best['direction']}), r={best['r']:+.4f}, p={best['p']:.4f}")

# =====================
# 5. Visualization
# =====================
fig, axes = plt.subplots(3, 1, figsize=(14, 16))
fig.suptitle('BTC Fear & Greed Index Analysis', fontsize=17, fontweight='bold')

# --- Chart 1: Price + FNG Trend ---
ax1 = axes[0]
ax2 = ax1.twinx()
ax1.plot(df['date'], df['close'], color='orange', label='BTC Price', linewidth=1.8)
ax2.fill_between(df['date'], df['fng_value'], alpha=0.25, color='steelblue')
ax2.plot(df['date'], df['fng_value'], color='steelblue', linewidth=1.0, label='FNG Index')
ax1.set_ylabel('BTC Price (USDT)', color='orange', fontsize=11)
ax2.set_ylabel('Fear & Greed Index', color='steelblue', fontsize=11)
ax2.axhline(y=25, color='red', linestyle='--', alpha=0.4, linewidth=0.8)
ax2.axhline(y=50, color='gray', linestyle='--', alpha=0.4, linewidth=0.8)
ax2.axhline(y=75, color='green', linestyle='--', alpha=0.4, linewidth=0.8)
ax2.set_ylim(0, 100)
ax1.set_title('BTC Price vs Fear & Greed Index', fontsize=13)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

# --- Chart 2: Boxplot by FNG Zone ---
def fng_label(v):
    if v <= 25: return 'Extreme Fear\n(0-25)'
    elif v <= 45: return 'Fear\n(26-45)'
    elif v <= 55: return 'Neutral\n(46-55)'
    elif v <= 75: return 'Greed\n(56-75)'
    else: return 'Extreme Greed\n(76-100)'

df['fng_zone'] = df['fng_value'].apply(fng_label)
zone_order = ['Extreme Fear\n(0-25)', 'Fear\n(26-45)', 'Neutral\n(46-55)', 'Greed\n(56-75)', 'Extreme Greed\n(76-100)']
palette = ['#d73027', '#fc8d59', '#fee090', '#91cf60', '#1a9850']

sns.boxplot(data=df, x='fng_zone', y='return_1d', order=zone_order,
            palette=palette, ax=axes[1], width=0.5)
axes[1].axhline(y=0, color='black', linewidth=1.0, linestyle='--')
axes[1].set_xlabel('Fear & Greed Zone', fontsize=11)
axes[1].set_ylabel('BTC 1D Return (%)', fontsize=11)
axes[1].set_title('BTC 1D Return Distribution by FNG Zone\n(Box: 25~75%, Line: Median)', fontsize=13)

for i, zone in enumerate(zone_order):
    mean_val = df[df['fng_zone'] == zone]['return_1d'].mean()
    count = len(df[df['fng_zone'] == zone])
    axes[1].text(i, axes[1].get_ylim()[1] * 0.85,
                 f'Mean\n{mean_val:+.2f}%\n(n={count})',
                 ha='center', fontsize=9, color='black')

# --- Chart 3: Lag Correlation Bar ---
colors = ['tomato' if p < 0.05 else 'steelblue' for p in df_lag['p']]
bars = axes[2].bar(df_lag['lag'], df_lag['r'], color=colors, alpha=0.8, edgecolor='white')
axes[2].axhline(y=0, color='black', linewidth=0.8)
axes[2].set_xlabel('Lag (days)   ←  FNG Leads  |  Price Leads  →', fontsize=11)
axes[2].set_ylabel('Pearson r', fontsize=11)
axes[2].set_title('Cross-Correlation: FNG -> BTC 1D Return\n(Red = p<0.05 Significant)', fontsize=13)
axes[2].set_xticks(list(lags))

for bar, row in zip(bars, lag_results):
    axes[2].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.002 if row['r'] >= 0 else bar.get_height() - 0.008,
                 f"{row['r']:+.3f}", ha='center', fontsize=8)

legend_elements = [Patch(facecolor='tomato', label='p < 0.05 (Significant)'),
                   Patch(facecolor='steelblue', label='p >= 0.05 (Not Significant)')]
axes[2].legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.savefig('fng_btc_analysis_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Chart saved: fng_btc_analysis_v2.png")